**Project Archeticture:**
User Question
      ↓
Prompt with DB schema
      ↓
LLM generates SQL
      ↓
Python executes SQL
      ↓
Database returns results
      ↓
Optional LLM explanation

We are using sqlite which doesnot need server setup, is lightweight and easy in Jupyter, and can be connected with jupyter as well
Sqlite3 is a python library for sql databases. So the library will help to create a database, run the queries and fetch results. 
Once we have an sql setup, we can use other sql databases that can be loaded intp jupyter by connecting the specific sql server.
                           

**Step 1: import libraries and create database connection**

In [2]:
import sqlite3
import pandas as pd

#create sql database connection object
conn = sqlite3.connect("sales.db")

In [4]:
#create the dataset- Now we create a small dataset using pandas - basically a dataframe with columns and rows

sales_data = pd.DataFrame({
    "product": [
        "Laptop",
        "Phone",
        "Tablet",
        "Laptop",
        "Phone"
    ],

    "sales": [
        1000,
        500,
        300,
        1200,
        700
    ],

    "region": [
        "East",
        "West",
        "East",
        "South",
        "North"
    ]
})

In [5]:
sales_data.head()

,product,sales,region
0,Laptop,1000,East
1,Phone,500,West
2,Tablet,300,East
3,Laptop,1200,South
4,Phone,700,North


In [6]:
sales_data.dtypes

product    object
sales       int64
region     object
dtype: object

In [7]:
sales_data.shape

(5, 3)

**Step 2: Store DataFrame into SQL Database**: the 'sales_data' currently exists only in oython memory as python dataframe, we need to put into actual sql table - 'sales_db' (in SQLite) we created while doing the connection-

- Take pandas DataFrame
-         ↓
- Convert into SQL table
-         ↓
- Store in database

In [10]:
# we are giving the sql table name as 'sales' where we are storing the dataframe after moving into sql
sales_data.to_sql("sales",conn,if_exists="replace",index=False) #index = false because we do not want to save the index column which came in within pandas df

5

In [15]:
#test where the table exists

query = "select * from sales"
query_result = pd.read_sql(query, conn) # the actual python execution of sql query
query_result
#so database executes the query, returns the result, converts it into dataframe and stores dataframe into query_result

,product,sales,region
0,Laptop,1000,East
1,Phone,500,West
2,Tablet,300,East
3,Laptop,1200,South
4,Phone,700,North


**The actual sql chatbot concept is, user should ask "Show total sales by region", instead of we writing "select * from...." and Gemini should generate the SQL.** 
- LLM/Gemini only generates the query text, and sql is executed by python

**Step 3: Create Prompt : Natural Language → SQL generation by LLM**
- As mentioned above, query is executed by python using pd.read_sql(user_query_translated by gemini into sql text, connection_name)
For Gemini to convert the user query to sql, we need to create a prompt which will have the table schema with columns, rows, data. etc...SInce LLM has no idea about what table and what column exists.SO we create the data structure insode the prompt for LLM to access it. This becomes a RAG prompt.

In [17]:
# lets create the schema, and user question

schema = """
Table name: sales

Columns:
product
sales
region
"""

user_question = "Show total sales by region"

**Load the LLM libraries, API key, model name before calling LLM functions**

In [21]:
import google.generativeai as genai

# same like openai, first generate/get the api key 
API_KEY = os.getenv("GOOGLE_API_KEY")
genai.configure(api_key=API_KEY)

# create a helper function if you are using any message or prompt
# then call the model name that you are using
model = genai.GenerativeModel("gemini-2.5-flash")

In [22]:
# the prompt
prompt = f"""
You are an expert SQL assistant.

Convert the following natural language question into SQL.

Database schema:
{schema}

Question:
{user_question}

Only return SQL query.
"""

# Gemini's response/translation of user question into sql
response = model.generate_content(prompt)

sql_query = response.text # storing the response into a variable

#print the query
print(sql_query)

```sql
SELECT region, SUM(sales) AS total_sales
FROM sales
GROUP BY region;
```


**Step 4: Execute the generated SQL automatically**
- Python needs to execute the query from database.

LLM usually returns the text with some syntax errors, example:
- extra formatting
- markdown
- explanations
- comments

So we fix the issue with below code before running the python execution

In [26]:
sql_query = sql_query.replace("```sql", "")
sql_query = sql_query.replace("```", "")

print(sql_query)


SELECT region, SUM(sales) AS total_sales
FROM sales
GROUP BY region;



In [27]:
query_result = pd.read_sql(sql_query, conn)
query_result

,region,total_sales
0,East,1300
1,North,700
2,South,1200
3,West,500


**Now the flow looks like:**

- User Question
      ↓
- Gemini generates SQL
      ↓
- Python cleans SQL
      ↓
- SQLite executes query
      ↓
- Results returned

**Step 5: Create 2 helper functions, 1st the SQL text generation function (done by LLM), and 2nd for query execution from database (done by python on SQLite)**                                                                                       

In [30]:
# function 1

def generate_sql(user_question, schema): # provide the user question and the schema for LLM

    prompt = f"""
    You are an expert SQL assistant.

    Convert the following natural language question into SQL.

    Database schema:
    {schema}

    Question:
    {user_question}

    Only return SQL query.
    """

    response = model.generate_content(prompt) #model generates the response 
    sql_query = response.text #response is stored in variable

    # usually the unwanted formatting part '''sql''' will occur twice, at the begining and at the end so we do the cleanup twice
    sql_query = sql_query.replace( # clean markdown formatting
        "```sql",
        "")

    sql_query = sql_query.replace( # clean markdown formatting
        "```",
        "")

    return sql_query

In [29]:
# Function 2

def run_sql_query(sql_query, conn): # pass the sql_query post formatting, and connection name

    result = pd.read_sql(sql_query, conn)

    return result

**Step 6: Similar to RAG ask_question() function, we create the final chatbot function whick makes the full AI pipeline reusable**,
- Basically this final chatbot function will incorporate/implement the above 2 fucntions in it to make the pipeline work.

In [31]:
def ask_database(user_question):

    # Step 1: Generate SQL, using the function1 we created above
    sql_query = generate_sql(user_question,schema)

    print("Generated SQL:")
    print(sql_query)

    # Step 2: Execute SQL, using funtion 2 we created above
    result = run_sql_query(
        sql_query,
        conn
    )

    return result

**Step 7: Convert into chatbot loop - create continous conversation**

In [ ]:
while True:

    user_question = input("Ask your database question: ")

    if user_question.lower() == "exit":
        break

    response = ask_database(user_question) # calling the final function we created above

    print("\nResult:")
    print(response)
    print("\n")